# cerberus-neuro — NeuroPainting data exploration (S3 audit)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PatrickJReed/cerberus-neuro/blob/main/notebooks/01_data_exploration.ipynb)

Verify the public NeuroPainting dataset on the Cell Painting Gallery (`s3://cellpainting-gallery/cpg0038-tegtmeyer-neuropainting/`) against the project dossier. The dossier covers the descriptive layer (cohort, cell types, dyes, total size). This notebook resolves the gaps the dossier itself flags as unverified, plus the implementation details we need before designing the data pipeline.

**Verification punch list:**

1. Channel count: 5 vs 6. Resume says 5; the NeuroPainting protocol uses 6 dyes (Hoechst, MitoTracker, Phalloidin, ConA, SYTO 14, WGA). Confirm whether channels 3 (Phalloidin) and 4 (ConA) are stored separately in the gallery or pre-combined.
2. Brightfield channel: how labeled in `load_data.csv` filenames relative to fluorescence?
3. Image dimensions in pixels (not in the publication).
4. Fields of view per well. Paper implies 1; gallery may have more.
5. Cohort: 44 lines (22 control + 22 deletion) vs the unverified “plus 4 isogenic” resume claim.
6. Metadata schema: which `Metadata_*` columns encode cell type, cell line, and disease state.
7. File format, on-disk volume per plate, and what a Colab-Free-friendly subset looks like.

Stage A is listing + metadata-only (no images). Stage B is gated and downloads a single well to inspect channel mapping and dimensions.

Run `00_environment_smoke.ipynb` first to confirm the runtime is wired up.


## 1. Setup

In [1]:
!pip install -q "cerberus-neuro @ git+https://github.com/PatrickJReed/cerberus-neuro.git@main"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 4.3 MB/s eta 0:00:00


In [2]:
import os
from pathlib import Path

try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get('HF_TOKEN'), add_to_git_credential=False)
    print('HF login: OK (Colab secret)')
except Exception as e:
    print(f'HF login skipped ({type(e).__name__}); not required for the public S3 audit')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = Path('/content/drive/MyDrive/cerberus-neuro/cache/audit_v0')
except Exception:
    CACHE_DIR = Path.home() / '.cache' / 'cerberus-neuro' / 'audit_v0'

CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f'Cache dir: {CACHE_DIR}')

HF login: OK (Colab secret)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cache dir: /content/drive/MyDrive/cerberus-neuro/cache/audit_v0


## 2. Stage A — listing + metadata-only audit

Anonymous (unsigned) S3 access via `boto3`. Cell Painting Gallery is a public Registry of Open Data bucket; no AWS account required.

In [ ]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config

BUCKET = 'cellpainting-gallery'
ROOT_PREFIX = 'cpg0038-tegtmeyer-neuropainting/'
WORKSPACE_PREFIX = ROOT_PREFIX + 'broad/workspace/'
IMAGES_PREFIX    = ROOT_PREFIX + 'broad/images/'

s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))

def list_level(prefix, max_keys=2000):
    """Return (sub_prefixes, keys_at_this_level) using delimiter='/'."""
    paginator = s3.get_paginator('list_objects_v2')
    dirs, keys = [], []
    for page in paginator.paginate(Bucket=BUCKET, Prefix=prefix, Delimiter='/', PaginationConfig={'PageSize': max_keys}):
        dirs.extend(p['Prefix'] for p in page.get('CommonPrefixes', []))
        keys.extend((o['Key'], o['Size']) for o in page.get('Contents', []))
    return dirs, keys

def list_recursive(prefix, max_keys=None):
    """Flat list of all (key, size) under prefix. Pass max_keys=None for unbounded."""
    paginator = s3.get_paginator('list_objects_v2')
    cfg = {'PageSize': 1000}
    if max_keys is not None:
        cfg['MaxItems'] = max_keys
    out = []
    for page in paginator.paginate(Bucket=BUCKET, Prefix=prefix, PaginationConfig=cfg):
        out.extend((o['Key'], o['Size']) for o in page.get('Contents', []))
    return out


### A1. Top-level prefix tree

Cell Painting Gallery datasets follow a `source_X / workspace / images` structure. Confirm what `cpg0038` actually has.

In [4]:
def show_tree(prefix, depth=2, indent=0):
    if indent > depth:
        return
    dirs, keys = list_level(prefix)
    pad = '  ' * indent
    for d in dirs:
        print(f'{pad}{d}')
        show_tree(d, depth=depth, indent=indent + 1)
    if indent == 0 and keys:
        print(f'{pad}({len(keys)} files at this level)')

show_tree(ROOT_PREFIX, depth=3)

cpg0038-tegtmeyer-neuropainting/broad/
  cpg0038-tegtmeyer-neuropainting/broad/images/
    cpg0038-tegtmeyer-neuropainting/broad/images/NCP_ASTROCYTES_1/
      cpg0038-tegtmeyer-neuropainting/broad/images/NCP_ASTROCYTES_1/illum/
      cpg0038-tegtmeyer-neuropainting/broad/images/NCP_ASTROCYTES_1/images/
      cpg0038-tegtmeyer-neuropainting/broad/images/NCP_ASTROCYTES_1/images_projected/
    cpg0038-tegtmeyer-neuropainting/broad/images/NCP_NEURONS_2_20x/
      cpg0038-tegtmeyer-neuropainting/broad/images/NCP_NEURONS_2_20x/illum/
      cpg0038-tegtmeyer-neuropainting/broad/images/NCP_NEURONS_2_20x/images/
      cpg0038-tegtmeyer-neuropainting/broad/images/NCP_NEURONS_2_20x/images_projected/
    cpg0038-tegtmeyer-neuropainting/broad/images/NCP_NEURONS_2_63x/
      cpg0038-tegtmeyer-neuropainting/broad/images/NCP_NEURONS_2_63x/illum/
      cpg0038-tegtmeyer-neuropainting/broad/images/NCP_NEURONS_2_63x/images/
      cpg0038-tegtmeyer-neuropainting/broad/images/NCP_NEURONS_2_63x/images_proj

### A2. Locate the metadata files

Cell Painting Gallery convention: per-plate `load_data.csv` / `load_data_with_illum.csv` lives under `workspace/load_data_csv/<batch>/<plate>/`, and dataset-level metadata (plate, well, compound) lives under `workspace/metadata/`.

In [ ]:
import re

# Workspace holds metadata, load_data, analysis, profiles. Small relative to images/.
# Listing the whole root would hit hundreds of thousands of TIFFs first and starve the CSV scan.
workspace_keys = list_recursive(WORKSPACE_PREFIX)
print(f'{len(workspace_keys):,} objects under {WORKSPACE_PREFIX}')

csv_keys = [(k, sz) for k, sz in workspace_keys if k.endswith(('.csv', '.csv.gz', '.tsv', '.tsv.gz'))]
print(f'{len(csv_keys):,} CSV/TSV objects')
for k, sz in csv_keys[:30]:
    print(f'  {sz:>10}  {k}')
if len(csv_keys) > 30:
    print(f'  ... +{len(csv_keys) - 30} more')


In [6]:
load_data_keys = [k for k, _ in csv_keys if 'load_data' in k.lower()]
metadata_keys  = [k for k, _ in csv_keys if '/metadata/' in k.lower()]
platemap_keys  = [k for k, _ in csv_keys if 'platemap' in k.lower() or 'plate_map' in k.lower()]

print('load_data.csv files:', len(load_data_keys))
for k in load_data_keys[:8]:
    print(' ', k)

print('\nmetadata/ files:', len(metadata_keys))
for k in metadata_keys[:20]:
    print(' ', k)

print('\nplate-map files:', len(platemap_keys))
for k in platemap_keys[:20]:
    print(' ', k)

load_data.csv files: 0

metadata/ files: 0

plate-map files: 0


In [7]:
import pandas as pd

def s3_to_local(key):
    local = CACHE_DIR / key.replace('/', '__')
    if not local.exists():
        s3.download_file(BUCKET, key, str(local))
    return local

sample_load = load_data_keys[0] if load_data_keys else None
print(f'Inspecting: {sample_load}')
if sample_load:
    p = s3_to_local(sample_load)
    df = pd.read_csv(p)
    print(f'shape: {df.shape}')
    print('columns:')
    for c in df.columns:
        print(f'  {c}')
    print('\nfirst row:')
    print(df.iloc[0])

Inspecting: None


### A3. Channel naming — 5 vs 6, brightfield label

Enumerate every `FileName_*` / `PathName_*` column. Each fluorescence + brightfield channel will appear once. The stain abbreviation in the column name (e.g. `OrigDNA`, `OrigMito`, `OrigAGP`, `OrigER`, `OrigRNA`, `OrigBrightfield`) tells us how the gallery has organized the channels.

In [8]:
if sample_load:
    file_cols = [c for c in df.columns if c.startswith('FileName_')]
    path_cols = [c for c in df.columns if c.startswith('PathName_')]
    print(f'channel count (FileName_ columns): {len(file_cols)}')
    for c in file_cols:
        print(f'  {c}  —  example: {df[c].iloc[0]}')
    print(f'\nPathName_ columns: {len(path_cols)}')
    for c in path_cols:
        print(f'  {c}  —  example: {df[c].iloc[0]}')

### A4. Cohort verification — lines, plates, wells, cell types

Pull the dataset-level metadata (plate / well / line manifests) and compare to the dossier:
- 44 lines (22 control + 22 deletion); the resume’s extra “+4 isogenic” is unverified.
- 4 cell types: iPSC, NPC, Neuron, Astrocyte.
- 8 wells per cell line (per the publication).

In [9]:
for k in metadata_keys + platemap_keys:
    p = s3_to_local(k)
    try:
        m = pd.read_csv(p)
    except Exception as e:
        print(f'skip {k}: {e}')
        continue
    print(f'\n=== {k}  shape={m.shape} ===')
    print('columns:', list(m.columns))
    print(m.head(3))

In [10]:
# Aggregate per-plate load_data.csv files to count plates / wells / sites / lines / cell types.
# Cell-type and disease-state encoding may live in Metadata_* columns of load_data, in a separate
# platemap, or in a well-level metadata CSV. Surface whatever is present.

frames = []
for k in load_data_keys:
    p = s3_to_local(k)
    try:
        d = pd.read_csv(p, usecols=lambda c: c.startswith('Metadata_') or c == 'Metadata_Site')
    except ValueError:
        d = pd.read_csv(p)
        d = d[[c for c in d.columns if c.startswith('Metadata_')]]
    d['_load_data_key'] = k
    frames.append(d)

meta = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(f'rows across load_data files: {len(meta)}')
print('Metadata_* columns observed:')
for c in [c for c in meta.columns if c.startswith('Metadata_')]:
    n = meta[c].nunique(dropna=True)
    sample = sorted(meta[c].dropna().astype(str).unique())[:6]
    print(f'  {c:40s} unique={n:5d}  e.g. {sample}')

rows across load_data files: 0
Metadata_* columns observed:


In [11]:
# Resolve cell-type / cell-line / disease-state columns by name pattern, then summarize.
import re as _re

def first_match(cols, *patterns):
    for p in patterns:
        for c in cols:
            if _re.search(p, c, _re.I):
                return c
    return None

cols = list(meta.columns)
col_plate     = first_match(cols, r'plate$', r'plate_id', r'plate')
col_well      = first_match(cols, r'well$', r'well_id', r'well')
col_site      = first_match(cols, r'site$', r'field')
col_line      = first_match(cols, r'line', r'donor', r'subject', r'cell_id')
col_celltype  = first_match(cols, r'cell.?type', r'differentiation', r'stage')
col_disease   = first_match(cols, r'genotype', r'disease', r'22q', r'condition', r'group')

for label, c in [('plate', col_plate), ('well', col_well), ('site/field', col_site),
                 ('cell line', col_line), ('cell type', col_celltype),
                 ('disease state', col_disease)]:
    if c:
        vals = sorted(meta[c].dropna().astype(str).unique())
        print(f'{label:14s} <- {c:30s}  n_unique={len(vals)}')
        if len(vals) <= 60:
            print('   values:', vals)
    else:
        print(f'{label:14s} <- NOT FOUND in load_data Metadata_* columns')

plate          <- NOT FOUND in load_data Metadata_* columns
well           <- NOT FOUND in load_data Metadata_* columns
site/field     <- NOT FOUND in load_data Metadata_* columns
cell line      <- NOT FOUND in load_data Metadata_* columns
cell type      <- NOT FOUND in load_data Metadata_* columns
disease state  <- NOT FOUND in load_data Metadata_* columns


### A5. Volume math — what does a Colab-Free subset look like?

Tally TIFFs and total bytes for one plate to estimate per-plate footprint, then extrapolate.

In [ ]:
# Per-batch image enumeration. Listing each batch separately keeps each request
# bounded and gives us per-batch counts for scoped-subset planning.
BATCHES = [
    'NCP_ASTROCYTES_1',
    'NCP_NEURONS_2_20x',
    'NCP_NEURONS_2_63x',
    'NCP_PROGENITORS_1',
    'NCP_STEM_1',
]

per_batch = {}
for b in BATCHES:
    keys = list_recursive(f'{IMAGES_PREFIX}{b}/images/')
    tiffs = [(k, sz) for k, sz in keys if k.lower().endswith(('.tif', '.tiff'))]
    n_tiff = len(tiffs)
    gb     = sum(sz for _, sz in tiffs) / 1e9
    per_batch[b] = {'n_tiff': n_tiff, 'gb': gb, 'n_objects': len(keys), 'tiffs': tiffs}
    print(f'{b:25s}  {n_tiff:>7,} tiffs  {gb:>7.1f} GB  ({len(keys):,} total objs)')

total_tiff = sum(v['n_tiff'] for v in per_batch.values())
total_gb   = sum(v['gb']     for v in per_batch.values())
print(f'\ntotal: {total_tiff:,} TIFFs, {total_gb:,.1f} GB')


## 3. Stage B — single-well image inspection (gated)

Gated behind `RUN_STAGE_B = True`. Downloads ~50–100 MB for one well and renders each channel.

Resolves: image dimensions in pixels, dtype / bit depth, fields-of-view per well, and visual confirmation of the channel-to-dye mapping inferred in A3.

In [ ]:
RUN_STAGE_B = False  # flip to True to download a single well

if RUN_STAGE_B and sample_load:
    one = df.iloc[0]
    file_cols = [c for c in df.columns if c.startswith('FileName_')]
    path_cols = [c for c in df.columns if c.startswith('PathName_')]
    pairs = []
    for fc in file_cols:
        stain = fc[len('FileName_'):]
        pc = f'PathName_{stain}'
        if pc in df.columns:
            # PathName_* in Cell Painting Gallery is usually a path relative to the bucket OR an s3 URI;
            # strip a leading scheme/bucket if present.
            path = str(one[pc])
            for pfx in (f's3://{BUCKET}/', '/'):
                if path.startswith(pfx):
                    path = path[len(pfx):]
            key = f"{path.rstrip('/')}/{one[fc]}"
            pairs.append((stain, key))
    for stain, key in pairs:
        print(stain, '->', key)
else:
    print('Stage B not run. Set RUN_STAGE_B = True to download one well.')

In [ ]:
if RUN_STAGE_B and sample_load:
    import tifffile
    import numpy as np
    import matplotlib.pyplot as plt

    well_dir = CACHE_DIR / 'well_sample'
    well_dir.mkdir(exist_ok=True)
    images = {}
    for stain, key in pairs:
        local = well_dir / Path(key).name
        if not local.exists():
            s3.download_file(BUCKET, key, str(local))
        arr = tifffile.imread(str(local))
        images[stain] = arr
        print(f'{stain:20s} shape={arr.shape}  dtype={arr.dtype}  min={arr.min()}  max={arr.max()}')

    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3))
    if n == 1:
        axes = [axes]
    for ax, (stain, arr) in zip(axes, images.items()):
        a = arr.astype(np.float32)
        lo, hi = np.percentile(a, [1, 99])
        a = np.clip((a - lo) / max(hi - lo, 1e-6), 0, 1)
        ax.imshow(a, cmap='gray')
        ax.set_title(stain, fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## 4. Audit summary

Capture the resolved facts for downstream pipeline design (`src/cerberus_neuro/data.py`).

In [ ]:
import json

audit = {
    'bucket': BUCKET,
    'root_prefix': ROOT_PREFIX,
    'workspace_prefix': WORKSPACE_PREFIX,
    'images_prefix': IMAGES_PREFIX,
    'batches': list(per_batch.keys()),
    'per_batch_summary': {b: {'n_tiff': v['n_tiff'], 'gb': round(v['gb'], 1)} for b, v in per_batch.items()},
    'total_tiff_count': total_tiff,
    'total_tiff_gb': round(total_gb, 1),
    'load_data_files': len(load_data_keys),
    'metadata_files': metadata_keys,
    'channel_columns': [c for c in (df.columns if sample_load else []) if c.startswith('FileName_')],
    'metadata_column_resolution': {
        'plate': col_plate,
        'well': col_well,
        'site': col_site,
        'cell_line': col_line,
        'cell_type': col_celltype,
        'disease_state': col_disease,
    },
}
out = CACHE_DIR / 'audit_v0_findings.json'
out.write_text(json.dumps(audit, indent=2, default=str))
print(f'wrote {out}')
print(json.dumps(audit, indent=2, default=str))
